# TN0 — dựng lại kết quả MobiVital

Giải thích từng bước: [TN0.md](TN0.md)

## 0. Setup

In [2]:
!pip install torch numpy scipy pandas einops tqdm

... (12 dong da an) ...


In [1]:
import os
import csv
import glob
import shutil

print(os.getcwd())

/Users/udnb/Desktop/THESIS_GRADUATE/notebooks


In [3]:
!python3 -c "import torch; print(torch.__version__, torch.cuda.is_available())"

2.8.0 False


In [4]:
files = [
    "/Users/udnb/Desktop/THESIS_GRADUATE/data/raw",
    "/Users/udnb/Desktop/THESIS_GRADUATE/data/processed/mobivital_original/training_breath_tripod_data.npy",
    "/Users/udnb/Desktop/THESIS_GRADUATE/external/mobivital/checkpoints/lstm_pred_tripod_0.9.pth",
    "/Users/udnb/Desktop/THESIS_GRADUATE/external/mobivital/inference/methods/tripod_mobivital_pre_invert_0.9.txt",
]

for path in files:
    print(os.path.exists(path), path)

True /Users/udnb/Desktop/THESIS_GRADUATE/data/raw
True /Users/udnb/Desktop/THESIS_GRADUATE/data/processed/mobivital_original/training_breath_tripod_data.npy
True /Users/udnb/Desktop/THESIS_GRADUATE/external/mobivital/checkpoints/lstm_pred_tripod_0.9.pth
True /Users/udnb/Desktop/THESIS_GRADUATE/external/mobivital/inference/methods/tripod_mobivital_pre_invert_0.9.txt


## 1. Dựng thư mục work

In [2]:
os.makedirs("/Users/udnb/Desktop/THESIS_GRADUATE/runs/tn0/work/dataset/mobivital/tripod", exist_ok=True)
os.makedirs("/Users/udnb/Desktop/THESIS_GRADUATE/runs/tn0/work/dataset_old_names/mobivital/tripod", exist_ok=True)
os.makedirs("/Users/udnb/Desktop/THESIS_GRADUATE/runs/tn0/work/data_final", exist_ok=True)
os.makedirs("/Users/udnb/Desktop/THESIS_GRADUATE/runs/tn0/work/checkpoints", exist_ok=True)
os.makedirs("/Users/udnb/Desktop/THESIS_GRADUATE/runs/tn0/work/inference/methods", exist_ok=True)

print(sorted(os.listdir("/Users/udnb/Desktop/THESIS_GRADUATE/runs/tn0/work")))

['checkpoints', 'data_final', 'dataset', 'dataset_old_names', 'inference']


### 1.1 Symlink 1874 file CSV

In [6]:
csv_files = glob.glob("/Users/udnb/Desktop/THESIS_GRADUATE/data/raw/*/*.csv")

for path in csv_files:
    name = os.path.basename(path)
    link = "/Users/udnb/Desktop/THESIS_GRADUATE/runs/tn0/work/dataset/mobivital/tripod/" + name
    if not os.path.lexists(link):
        os.symlink(path, link)

print(len(os.listdir("/Users/udnb/Desktop/THESIS_GRADUATE/runs/tn0/work/dataset/mobivital/tripod")))

1874


### 1.2 Thư mục thứ hai, thêm 52 tên cũ

In [3]:
for name in os.listdir("/Users/udnb/Desktop/THESIS_GRADUATE/runs/tn0/work/dataset/mobivital/tripod"):
    link = "/Users/udnb/Desktop/THESIS_GRADUATE/runs/tn0/work/dataset_old_names/mobivital/tripod/" + name
    real = os.path.realpath("/Users/udnb/Desktop/THESIS_GRADUATE/runs/tn0/work/dataset/mobivital/tripod/" + name)
    if not os.path.lexists(link):
        os.symlink(real, link)

print(len(os.listdir("/Users/udnb/Desktop/THESIS_GRADUATE/runs/tn0/work/dataset_old_names/mobivital/tripod")))

1874


In [4]:
rows = csv.reader(open("/Users/udnb/Desktop/THESIS_GRADUATE/external/mobivital/inference/methods/tripod_mobivital_pre_invert_0.9.txt"))
names_in_txt = []
for row in rows:
    names_in_txt.append(row[0])

count = 0
for old_name in names_in_txt:
    link = "/Users/udnb/Desktop/THESIS_GRADUATE/runs/tn0/work/dataset_old_names/mobivital/tripod/" + old_name
    if os.path.lexists(link):
        continue
    new_name = old_name[:2] + "12" + old_name[4:]
    real = os.path.realpath("/Users/udnb/Desktop/THESIS_GRADUATE/runs/tn0/work/dataset_old_names/mobivital/tripod/" + new_name)
    os.symlink(real, link)
    count = count + 1

print("added", count)
print(len(os.listdir("/Users/udnb/Desktop/THESIS_GRADUATE/runs/tn0/work/dataset_old_names/mobivital/tripod")))

added 52
1926


In [5]:
missing = 0
for name in names_in_txt:
    if not os.path.exists("/Users/udnb/Desktop/THESIS_GRADUATE/runs/tn0/work/dataset_old_names/mobivital/tripod/" + name):
        missing = missing + 1

print("names in txt", len(names_in_txt))
print("missing     ", missing, "  -> 0")
print("gen folder  ", len(os.listdir("/Users/udnb/Desktop/THESIS_GRADUATE/runs/tn0/work/dataset/mobivital/tripod")), "  -> 1874")

names in txt 537
missing      0   -> 0
gen folder   1874   -> 1874


### 1.3 Symlink 2 file .npy

In [10]:
for name in ["training_breath_tripod_data.npy", "testing_breath_tripod_data.npy"]:
    link = "/Users/udnb/Desktop/THESIS_GRADUATE/runs/tn0/work/data_final/" + name
    if not os.path.lexists(link):
        os.symlink("/Users/udnb/Desktop/THESIS_GRADUATE/data/processed/mobivital_original/" + name, link)

print(os.listdir("/Users/udnb/Desktop/THESIS_GRADUATE/runs/tn0/work/data_final"))

['training_breath_tripod_data.npy', 'testing_breath_tripod_data.npy']


### 1.4 Copy checkpoint và file TXT

In [6]:
shutil.copy("/Users/udnb/Desktop/THESIS_GRADUATE/external/mobivital/checkpoints/lstm_pred_tripod_0.9.pth",
            "/Users/udnb/Desktop/THESIS_GRADUATE/runs/tn0/work/checkpoints/lstm_pred_tripod_0.9.pth")

shutil.copy("/Users/udnb/Desktop/THESIS_GRADUATE/external/mobivital/checkpoints/optimal_params.json",
            "/Users/udnb/Desktop/THESIS_GRADUATE/runs/tn0/work/checkpoints/optimal_params.json")

shutil.copy("/Users/udnb/Desktop/THESIS_GRADUATE/external/mobivital/inference/methods/tripod_mobivital_pre_invert_0.9.txt",
            "/Users/udnb/Desktop/THESIS_GRADUATE/runs/tn0/work/inference/methods/TN0a.txt")

print(os.listdir("/Users/udnb/Desktop/THESIS_GRADUATE/runs/tn0/work/checkpoints"))
print(os.listdir("/Users/udnb/Desktop/THESIS_GRADUATE/runs/tn0/work/inference/methods"))

['optimal_params.json', 'lstm_pred_tripod_0.9.pth', 'lstm_retrained_tripod_0.9.pth']
['TN0b.txt', 'TN0c.txt', 'TN0a.txt', 'scores_TN0c.csv', 'scores_TN0b.csv']


## 2. TN0a

In [7]:
!cd /Users/udnb/Desktop/THESIS_GRADUATE/runs/tn0/work && \
PYTHONPATH=/Users/udnb/Desktop/THESIS_GRADUATE/external/mobivital \
python3 /Users/udnb/Desktop/THESIS_GRADUATE/external/mobivital/inference/evaluate.py \
-m TN0a.txt -d ./dataset_old_names/mobivital/tripod/ --save_file scores_TN0a.csv

0.8194811075582306
537it [00:42, 12.78it/s]


## 3. TN0b

In [13]:
print(os.path.exists("/Users/udnb/Desktop/THESIS_GRADUATE/runs/tn0/work/inference/methods/tripod_mobivital_pre_invert_0.9.txt"), " -> False")

False  -> False


In [14]:
!cd /Users/udnb/Desktop/THESIS_GRADUATE/runs/tn0/work && \
PYTHONPATH=/Users/udnb/Desktop/THESIS_GRADUATE/external/mobivital \
python3 /Users/udnb/Desktop/THESIS_GRADUATE/external/mobivital/inference/mobivital_gen.py

0.8221751511496864
100%|██████████████████████████████████████| 1874/1874 [41:50<00:00,  1.34s/it]


In [15]:
shutil.move("/Users/udnb/Desktop/THESIS_GRADUATE/runs/tn0/work/inference/methods/tripod_mobivital_pre_invert_0.9.txt",
            "/Users/udnb/Desktop/THESIS_GRADUATE/runs/tn0/work/inference/methods/TN0b.txt")

print(len(open("/Users/udnb/Desktop/THESIS_GRADUATE/runs/tn0/work/inference/methods/TN0b.txt").readlines()), " -> 537")

537  -> 537


In [5]:
!cd /Users/udnb/Desktop/THESIS_GRADUATE/runs/tn0/work && \
PYTHONPATH=/Users/udnb/Desktop/THESIS_GRADUATE/external/mobivital \
python3 /Users/udnb/Desktop/THESIS_GRADUATE/external/mobivital/inference/evaluate.py \
-m TN0b.txt --save_file scores_TN0b.csv

0.8221751511496864
537it [00:40, 13.17it/s]


### 3.1 So từng dòng TN0a với TN0b

In [17]:
def read_choices(path):
    table = {}
    for row in csv.reader(open(path)):
        table[row[0]] = (row[1], row[2])
    return table

choices_a = read_choices("/Users/udnb/Desktop/THESIS_GRADUATE/runs/tn0/work/inference/methods/TN0a.txt")
choices_b = read_choices("/Users/udnb/Desktop/THESIS_GRADUATE/runs/tn0/work/inference/methods/TN0b.txt")

same = 0
for old_name in choices_a:
    new_name = old_name[:2] + "12" + old_name[4:]
    if choices_b.get(old_name) == choices_a[old_name]:
        same = same + 1
    elif choices_b.get(new_name) == choices_a[old_name]:
        same = same + 1

print(same, "/", len(choices_a))

286 / 537


## 4. TN0c

In [18]:
!cd /Users/udnb/Desktop/THESIS_GRADUATE/runs/tn0/work && \
PYTHONPATH=/Users/udnb/Desktop/THESIS_GRADUATE/external/mobivital \
python3 /Users/udnb/Desktop/THESIS_GRADUATE/external/mobivital/training/autoreg_training.py \
--model_name lstm_retrained

{'epochs': 20, 'lr': 0.0001, 'batch_size': 64, 'hidden_size': 352, 'history_length': 200, 'future_length': 25, 'num_layers': 2, 'params_file': 'checkpoints/optimal_params.json', 'data_folder': './data_final', 'save_folder': './checkpoints', 'model_name': 'lstm_retrained', 'mode': 'tripod', 'corr_threshold': 0.9, 'top': False, 'device': 0}
/Users/udnb/Desktop/THESIS_GRADUATE/.venv/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
warnings.warn(warn_msg)
Epoch:0 loss: 0.037
Epoch:10 loss: 0.016
Finished Training
100%|██████████████████████████████████████| 20/20 [6:22:08<00:00, 1146.41s/it]


In [19]:
print(os.listdir("/Users/udnb/Desktop/THESIS_GRADUATE/runs/tn0/work/checkpoints"))
print(os.path.exists("/Users/udnb/Desktop/THESIS_GRADUATE/runs/tn0/work/inference/methods/tripod_mobivital_pre_invert_0.9.txt"), " -> False")

['optimal_params.json', 'lstm_pred_tripod_0.9.pth', 'lstm_retrained_tripod_0.9.pth']
False  -> False


In [20]:
!cd /Users/udnb/Desktop/THESIS_GRADUATE/runs/tn0/work && \
PYTHONPATH=/Users/udnb/Desktop/THESIS_GRADUATE/external/mobivital \
python3 /Users/udnb/Desktop/THESIS_GRADUATE/external/mobivital/inference/mobivital_gen.py \
--model_name lstm_retrained

0.7987479281268859
100%|██████████████████████████████████████| 1874/1874 [40:11<00:00,  1.29s/it]


In [21]:
shutil.move("/Users/udnb/Desktop/THESIS_GRADUATE/runs/tn0/work/inference/methods/tripod_mobivital_pre_invert_0.9.txt",
            "/Users/udnb/Desktop/THESIS_GRADUATE/runs/tn0/work/inference/methods/TN0c.txt")

print(len(open("/Users/udnb/Desktop/THESIS_GRADUATE/runs/tn0/work/inference/methods/TN0c.txt").readlines()), " -> 537")

537  -> 537


In [6]:
!cd /Users/udnb/Desktop/THESIS_GRADUATE/runs/tn0/work && \
PYTHONPATH=/Users/udnb/Desktop/THESIS_GRADUATE/external/mobivital \
python3 /Users/udnb/Desktop/THESIS_GRADUATE/external/mobivital/inference/evaluate.py \
-m TN0c.txt --save_file scores_TN0c.csv

0.7987479281268859
537it [00:41, 13.07it/s]


## 5. Kết quả

In [7]:
for name in sorted(os.listdir("/Users/udnb/Desktop/THESIS_GRADUATE/runs/tn0/work/inference/methods")):
    path = "/Users/udnb/Desktop/THESIS_GRADUATE/runs/tn0/work/inference/methods/" + name
    print(name, len(open(path).readlines()))

TN0a.txt 537
TN0b.txt 537
TN0c.txt 537
scores_TN0a.csv 538
scores_TN0b.csv 538
scores_TN0c.csv 538


In [2]:
import pandas as pd

a = pd.read_csv("/Users/udnb/Desktop/THESIS_GRADUATE/runs/tn0/work/inference/methods/scores_TN0a.csv", index_col=0)["TN0a.txt"]
b = pd.read_csv("/Users/udnb/Desktop/THESIS_GRADUATE/runs/tn0/work/inference/methods/scores_TN0b.csv", index_col=0)["TN0b.txt"]
c = pd.read_csv("/Users/udnb/Desktop/THESIS_GRADUATE/runs/tn0/work/inference/methods/scores_TN0c.csv", index_col=0)["TN0c.txt"]

print("TN0a  %d session  mean %.6f" % (len(a), a.mean()))
print("TN0b  %d session  mean %.6f" % (len(b), b.mean()))
print("TN0c  %d session  mean %.6f" % (len(c), c.mean()))

TN0a  537 session  mean 0.819481
TN0b  537 session  mean 0.822175
TN0c  537 session  mean 0.798748


In [3]:
def mean_by_user(scores):
    total = {}
    count = {}
    for name in scores.index:
        user = name.split("_")[1][-1]
        total[user] = total.get(user, 0) + scores[name]
        count[user] = count.get(user, 0) + 1
    result = {}
    for user in sorted(total):
        result[user] = total[user] / count[user]
    return result

mean_a = mean_by_user(a)
mean_b = mean_by_user(b)
mean_c = mean_by_user(c)

print("user   TN0a     TN0b     TN0c")
for user in sorted(mean_a):
    print("%s      %.4f   %.4f   %.4f" % (user, mean_a[user], mean_b[user], mean_c[user]))

user   TN0a     TN0b     TN0c
G      0.9226   0.9173   0.9176
H      0.6907   0.6819   0.6347
I      0.7658   0.7775   0.7594
J      0.9173   0.9313   0.9022


### 5.1 Điểm macro theo người (docs/PROTOCOL.md mục 4)

In [4]:
import numpy as np

print("        micro    macro    std")
for ten, diem, trung_binh_nguoi in [("TN0a", a, mean_a), ("TN0b", b, mean_b), ("TN0c", c, mean_c)]:
    bon_nguoi = []
    for user in sorted(trung_binh_nguoi):
        bon_nguoi.append(trung_binh_nguoi[user])
    bon_nguoi = np.array(bon_nguoi)
    print("%s    %.4f   %.4f   %.4f" % (ten, diem.mean(), bon_nguoi.mean(), bon_nguoi.std()))

micro    macro    std
TN0a    0.8195   0.8241   0.0995
TN0b    0.8222   0.8270   0.1031
TN0c    0.7987   0.8035   0.1153
